# 05. Hyperparameter Tuning

Tune each model's hyperparameters using patient-grouped cross-validation.

## Setup

In [ ]:
import sys
sys.path.append('..')

from config import config

## 1.Setup and load training data

In [1]:

import json
import sys
from pathlib import Path

import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedGroupKFold,
)

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import config

train_df = pd.read_csv(config.PROCESSED_TRAIN_FILE)
val_df = pd.read_csv(config.PROCESSED_VAL_FILE)

feature_cols = [
    column
    for column in train_df.columns
    if column.startswith(("F_", "U_", "FU_"))
]

assert len(feature_cols) == 103

X_train = train_df[feature_cols]
y_train = train_df[config.TARGET_COLUMN]
train_groups = train_df[config.PATIENT_ID_COLUMN]

X_val = val_df[feature_cols]
y_val = val_df[config.TARGET_COLUMN]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Training patients:", train_groups.nunique())

Training: (14540, 103)
Validation: (3115, 103)
Training patients: 8742


## 2.Define the tuning search space


In [2]:
base_random_forest = RandomForestClassifier(
    random_state=config.RANDOM_STATE,
    n_jobs=1
)

parameter_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.5],
    "class_weight": [None, "balanced"],
    "bootstrap": [True],
}

group_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=config.RANDOM_STATE
)

random_search = RandomizedSearchCV(
    estimator=base_random_forest,
    param_distributions=parameter_distributions,
    n_iter=20,
    scoring="f1",
    cv=group_cv,
    refit=True,
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
)

print("Randomized search configured.")
print("Candidates:", random_search.n_iter)
print("CV folds:", group_cv.n_splits)
print("Total fits:", random_search.n_iter * group_cv.n_splits)

Randomized search configured.
Candidates: 20
CV folds: 5
Total fits: 100


## 3.Run patient-grouped tuning

In [3]:
random_search.fit(
    X_train,
    y_train,
    groups=train_groups
)

print("Tuning completed.")
print("Best cross-validation F1:")
print(f"{random_search.best_score_ * 100:.2f}%")

print("\nBest parameters:")
for parameter, value in random_search.best_params_.items():
    print(f"{parameter}: {value}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Tuning completed.
Best cross-validation F1:
90.13%

Best parameters:
n_estimators: 300
min_samples_split: 10
min_samples_leaf: 2
max_features: 0.5
max_depth: None
class_weight: balanced
bootstrap: True


## 4.Tuned validation metrics

In [4]:
tuned_random_forest = random_search.best_estimator_

tuned_val_predictions = tuned_random_forest.predict(X_val)

tuned_validation_metrics = {
    "accuracy": accuracy_score(
        y_val,
        tuned_val_predictions
    ),
    "abnormal_precision": precision_score(
        y_val,
        tuned_val_predictions,
        pos_label=1
    ),
    "abnormal_recall": recall_score(
        y_val,
        tuned_val_predictions,
        pos_label=1
    ),
    "abnormal_f1": f1_score(
        y_val,
        tuned_val_predictions,
        pos_label=1
    ),
}

tuned_metrics_df = pd.Series(
    {
        metric: value * 100
        for metric, value in tuned_validation_metrics.items()
    },
    name="Tuned Random Forest (%)"
).to_frame()

display(tuned_metrics_df.round(2))

,Tuned Random Forest (%)
accuracy,89.09
abnormal_precision,88.84
abnormal_recall,91.17
abnormal_f1,89.99


## 5.Confusion matrix and report

In [5]:
tuned_confusion = confusion_matrix(
    y_val,
    tuned_val_predictions,
    labels=[0, 1]
)

tuned_confusion_df = pd.DataFrame(
    tuned_confusion,
    index=[
        "Actual normal (0)",
        "Actual abnormal (1)"
    ],
    columns=[
        "Predicted normal (0)",
        "Predicted abnormal (1)"
    ]
)

display(tuned_confusion_df)

print(
    classification_report(
        y_val,
        tuned_val_predictions,
        labels=[0, 1],
        target_names=[
            "normal (0)",
            "abnormal (1)"
        ],
        digits=4
    )
)

,Predicted normal (0),Predicted abnormal (1)
Actual normal (0),1247,192
Actual abnormal (1),148,1528


              precision    recall  f1-score   support

  normal (0)     0.8939    0.8666    0.8800      1439
abnormal (1)     0.8884    0.9117    0.8999      1676

    accuracy                         0.8909      3115
   macro avg     0.8911    0.8891    0.8900      3115
weighted avg     0.8909    0.8909    0.8907      3115



## 6.Baseline vs tuned comparison

In [6]:
baseline_metrics_path = (
    config.METRICS_DIR
    / "random_forest_baseline_validation.json"
)

with baseline_metrics_path.open(
    "r",
    encoding="utf-8"
) as file:
    baseline_record = json.load(file)

baseline_metrics = baseline_record["validation_metrics"]

tuning_comparison_df = pd.DataFrame({
    "Baseline Random Forest (%)": {
        metric: value * 100
        for metric, value in baseline_metrics.items()
    },
    "Tuned Random Forest (%)": {
        metric: value * 100
        for metric, value in tuned_validation_metrics.items()
    },
})

tuning_comparison_df["Difference (percentage points)"] = (
    tuning_comparison_df["Tuned Random Forest (%)"]
    - tuning_comparison_df["Baseline Random Forest (%)"]
)

display(tuning_comparison_df.round(2))

,Baseline Random Forest (%),Tuned Random Forest (%),Difference (percentage points)
accuracy,88.67,89.09,0.42
abnormal_precision,88.75,88.84,0.09
abnormal_recall,90.39,91.17,0.78
abnormal_f1,89.57,89.99,0.42


## 7.save the tuning results

In [7]:
config.METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

tuned_report_dict = classification_report(
    y_val,
    tuned_val_predictions,
    labels=[0, 1],
    target_names=[
        "normal (0)",
        "abnormal (1)"
    ],
    output_dict=True
)

tuned_record = {
    "experiment": "tuned_random_forest_all_103_features",
    "random_seed": config.RANDOM_STATE,
    "training_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "feature_count": len(feature_cols),
    "target": config.TARGET_COLUMN,
    "positive_class": 1,
    "scaling": None,
    "outlier_handling": None,
    "feature_selection": None,
    "search_method": "RandomizedSearchCV",
    "search_iterations": random_search.n_iter,
    "cross_validation": "5-fold StratifiedGroupKFold",
    "group_column": config.PATIENT_ID_COLUMN,
    "best_cross_validation_f1": float(
        random_search.best_score_
    ),
    "best_parameters": random_search.best_params_,
    "validation_metrics": tuned_validation_metrics,
    "confusion_matrix_labels_0_1": (
        tuned_confusion.tolist()
    ),
    "classification_report": tuned_report_dict,
}

tuned_metrics_path = (
    config.METRICS_DIR
    / "random_forest_tuned_validation.json"
)

with tuned_metrics_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        tuned_record,
        file,
        indent=2
    )

print("Saved tuned metrics:", tuned_metrics_path)

Saved tuned metrics: C:\Users\ASUS\Desktop\ML\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\results\metrics\random_forest_tuned_validation.json


## 8.save the Validation predictions

In [8]:
tuned_predictions_path = (
    config.METRICS_DIR
    / "random_forest_tuned_validation_predictions.csv"
)

tuned_predictions_df = val_df[
    [
        config.RECORD_ID_COLUMN,
        config.PATIENT_ID_COLUMN,
        config.TARGET_COLUMN
    ]
].copy()

tuned_predictions_df[
    "predicted_figo_label"
] = tuned_val_predictions

tuned_predictions_df.to_csv(
    tuned_predictions_path,
    index=False
)

print(
    "Saved tuned validation predictions:",
    tuned_predictions_path
)

Saved tuned validation predictions: C:\Users\ASUS\Desktop\ML\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\results\metrics\random_forest_tuned_validation_predictions.csv


## 9.save the full tuning search results

In [9]:
cv_results_df = pd.DataFrame(
    random_search.cv_results_
).sort_values(
    by="rank_test_score"
)

cv_results_path = (
    config.METRICS_DIR
    / "random_forest_tuning_cv_results.csv"
)

cv_results_df.to_csv(
    cv_results_path,
    index=False
)

print(
    "Saved cross-validation results:",
    cv_results_path
)

Saved cross-validation results: C:\Users\ASUS\Desktop\ML\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\results\metrics\random_forest_tuning_cv_results.csv


## 10.save the baseline comparison

In [10]:
tuning_comparison_path = (
    config.METRICS_DIR
    / "random_forest_baseline_vs_tuned.csv"
)

tuning_comparison_df.to_csv(
    tuning_comparison_path
)

print(
    "Saved baseline comparison:",
    tuning_comparison_path
)

Saved baseline comparison: C:\Users\ASUS\Desktop\ML\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\results\metrics\random_forest_baseline_vs_tuned.csv
